# 🎓 BÀI THỰC HÀNH TƯƠNG TÁC GOOGLE COLAB: CHƯƠNG 6
## GIÁM SÁT VỚI TENSORBOARD & QUẢN LÝ TRỌNG SỐ MÔ HÌNH
> **Người thực hiện:** Phạm Ngọc Phước  
> **Môn học:** Deep Learning (Học Sâu) - UTH  
> **Mục tiêu:** Chia nhỏ bài học thành từng phần tương tác để hiểu sâu cách hoạt động của lớp `TensorBoardLogger`.

---
### 📌 Lộ Trình 6 Phần Học:
1. **Phần 0:** Cài đặt môi trường & nạp thư viện
2. **Phần 1:** Kích hoạt TensorBoard nhúng trực tiếp trong Colab
3. **Phần 2:** Nạp dữ liệu CIFAR-10 và khởi tạo `TensorBoardLogger`
4. **Phần 3:** Học về `add_graph` (Trực quan hóa sơ đồ mạng ResNet-18)
5. **Phần 4:** Học về `add_image_grid` (Khử chuẩn hóa & hiển thị ảnh mẫu)
6. **Phần 5:** Học về `add_scalar` & `add_scalars` (Vẽ đồ thị Loss & Accuracy)
7. **Phần 6:** Học về `save_best_model` & `load_checkpoint` (Quản lý trọng số .pth)

---
## 🔹 PHẦN 0: CÀI ĐẶT MÔI TRƯỜNG & TẢI DỰ ÁN
Nếu bạn đang chạy trên **Google Colab**, ô code này sẽ tự động tải dự án từ GitHub về và cấu hình đường dẫn.

In [ ]:
import os, sys
from pathlib import Path

# Kiểm tra môi trường thực thi
IN_COLAB = 'google.colab' in sys.modules

if IN_COLAB:
    print('⚡ Đang chạy trên Google Colab!')
    if not os.path.exists('repo'):
        !git clone https://github.com/Tanlee29-01/Practice-2---Hands-on-practice-with-pre-trained-neural-network-architectures.git repo
    %cd repo
    sys.path.append(os.getcwd())
else:
    print('💻 Đang chạy trên máy tính cục bộ!')
    PROJECT_ROOT = Path(os.getcwd()).parent if 'notebooks' in os.getcwd() else Path(os.getcwd())
    sys.path.append(str(PROJECT_ROOT))

print('✅ Cấu hình môi trường hoàn tất!')

---
## 🔹 PHẦN 1: KÍCH HOẠT TENSORBOARD TRỰC TIẾP TRONG COLAB
Lệnh ma thuật `%load_ext tensorboard` cho phép mở giao diện trực quan ngay bên dưới cell này mà không cần mở tab mới!

In [ ]:
%load_ext tensorboard
%tensorboard --logdir runs

---
## 🔹 PHẦN 2: NẠP DỮ LIỆU CIFAR-10 & KHỞI TẠO LOGGER
Chúng ta sẽ nạp một lô ảnh thực tế từ CIFAR-10 và khởi tạo đối tượng `TensorBoardLogger` của Phước.

In [ ]:
import torch
from src.data_module import CIFAR10DataModule
from src.model_builder import TransferModelBuilder
from src.logger import TensorBoardLogger

# 1. Nạp dữ liệu mẫu
data_module = CIFAR10DataModule(data_dir='./data', batch_size=16, num_workers=0, image_size=224)
data_module.setup()
train_loader = data_module.train_dataloader()
sample_images, sample_labels = next(iter(train_loader))
print(f'✅ Batch ảnh mẫu: {sample_images.shape}')

# 2. Khởi tạo Logger
logger = TensorBoardLogger(experiment_name='colab_demo_phuoc')
print(f'✅ Logger sẵn sàng tại: {logger.log_dir}')

---
## 🔹 PHẦN 3: HỌC VỀ `add_graph` (SƠ ĐỒ KIẾN TRÚC MẠNG RESNET-18)
* **Mục đích:** Khảo sát luồng dữ liệu truyền qua 18 tầng nơ-ron từ đầu vào ảnh đến 10 lớp đầu ra.

In [ ]:
# Khởi tạo mô hình ResNet-18 Transfer Learning
builder = TransferModelBuilder(model_name='resnet18', num_classes=10, pretrained=False)
model = builder.build()

# Ghi đồ thị Computational Graph vào TensorBoard
logger.add_graph(model, sample_images)
logger.flush()
print('✅ [add_graph] Đã ghi Computational Graph!')
print('👉 Cuộn lên khung TensorBoard ở Phần 1, bấm tab GRAPHS để xem sơ đồ tương tác!')

---
## 🔹 PHẦN 4: HỌC VỀ `add_image_grid` (KHỬ CHUẨN HÓA & HIỂN THỊ ẢNH)
* **Công thức toán:** Tự động đảo ngược chuẩn hóa ImageNet để đưa ảnh về đúng dải màu RGB gốc $[0, 1]$:
$$X_{\text{original}} = X_{\text{normalized}} \times \sigma + \mu$$

In [ ]:
# Ghép lưới 16 ảnh CIFAR-10 mẫu (4x4) và ghi vào TensorBoard
logger.add_image_grid('CIFAR10_Samples/Batch_16', sample_images, nrow=4, unnormalize=True)
logger.flush()
print('✅ [add_image_grid] Đã ghi lưới ảnh mẫu thành công!')
print('👉 Cuộn lên khung TensorBoard ở Phần 1, bấm tab IMAGES để ngắm lưới ảnh rõ nét!')

---
## 🔹 PHẦN 5: HỌC VỀ `add_scalar` & `add_scalars` (VẼ ĐỒ THỊ LOSS & ACCURACY)
* **Mục đích:** Theo dõi độ hội tụ của mô hình và kiểm tra xem có bị quá khớp (Overfitting) hay không.

In [ ]:
# Mô phỏng số liệu thực nghiệm qua 3 epoch từ báo cáo
history_data = [
    {'epoch': 1, 'train_loss': 1.8210, 'train_acc': 45.0, 'val_loss': 0.8589, 'val_acc': 69.8},
    {'epoch': 2, 'train_loss': 0.6542, 'train_acc': 80.2, 'val_loss': 0.6322, 'val_acc': 77.2},
    {'epoch': 3, 'train_loss': 0.3811, 'train_acc': 87.0, 'val_loss': 0.6061, 'val_acc': 77.8},
]

for item in history_data:
    ep = item['epoch']
    logger.log_epoch(
        epoch=ep,
        train_loss=item['train_loss'],
        train_acc=item['train_acc'],
        test_loss=item['val_loss'],
        test_acc=item['val_acc'],
    )
    # Thuật toán lưu checkpoint khi đạt kỷ lục điểm số
    logger.save_best_model(model=model, metric=item['val_acc'], epoch=ep, higher_is_better=True)

logger.flush()
print('✅ [log_epoch] Đã ghi nhận toàn bộ chỉ số qua các epoch!')
print('👉 Cuộn lên khung TensorBoard ở Phần 1, bấm tab SCALARS để xem biểu đồ đường cong!')

---
## 🔹 PHẦN 6: HỌC VỀ `load_checkpoint` (KHÔI PHỤC TRỌNG SỐ MÔ HÌNH)
* Nạp lại trọng số từ file `.pth` vào mô hình để suy luận hoặc đánh giá.

In [ ]:
# Đường dẫn checkpoint tốt nhất
best_checkpoint_path = logger.checkpoint_dir / 'colab_demo_phuoc_best.pth'

# Khởi tạo mô hình rỗng mới
restored_model = builder.build()

# Nạp trọng số
ckpt = logger.load_checkpoint(restored_model, best_checkpoint_path)
print(f'✅ Đã khôi phục thành công mô hình từ: {best_checkpoint_path}')
print(f'📊 Kỷ lục ghi nhận trong checkpoint: {ckpt.get("metrics")}')

# Đóng logger
logger.close()
print('🎉 CHÚC MỪNG PHƯỚC ĐÃ HOÀN THÀNH TOÀN BỘ BÀI THỰC HÀNH CHƯƠNG 6 TRÊN COLAB!')